# Data across sessions

In [ ]:
import datajoint as dj
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import matplotlib as mpl
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import itertools
import pandas as pd
import os
from IPython.display import display, HTML
from scipy.stats import sem
import bisect


# os.environ['DJ_PASSWORD'] = 'Hunt4D4t4!'
# dj.config['database.password'] = os.getenv('DJ_PASSWORD')

# Style of plots
small_size=8
medium_size=20
large_size=40
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['pdf.fonttype'] =42
mpl.rcParams['axes.grid']=False
mpl.rcParams['axes.titlesize']=small_size
mpl.rcParams["figure.titlesize"] = medium_size
mpl.rcParams['axes.labelsize']=small_size
mpl.rcParams['xtick.labelsize']=small_size
mpl.rcParams['ytick.labelsize']=small_size
mpl.rcParams['legend.fontsize']=small_size


dj.config["enable_python_native_blobs"] = True
dj.config['database.host'] = 'database.eflab.org:3306'
dj.config['database.user'] = 'eflab'


schemata = {'experiment'   : 'lab_experiments',
            'stimulus'     : 'lab_stimuli',
            'behavior'     : 'lab_behavior',
            'interface'    : 'lab_interface',
            'mice'         : 'lab_mice'
           }

# create a virtual module for every database schema that you are going to use
for schema, value in schemata.items():
    globals()[schema] = dj.create_virtual_module(schema, value, create_tables=True, create_schema=True)

# Alias schemata
exp = experiment
stim = stimulus
beh = behavior
inter = interface

In [ ]:
animal_id = 299
from_session = 500
to_session = 800

In [ ]:
# ================================== Session filtering options (use date or session) ==================================
# Choose one or both filters
use_date_filter = False
use_session_filter = True

# Date range (if applicable)
from_date =  "2025-10-29" # for 299: -8-22 | for 288: -8-28 | for 326: -9-8
to_date =  "2025-10-31"

# Session range (if applicable)
from_session = from_session
to_session = to_session

key = {'animal_id': animal_id}

restr = exp.Session() & {'animal_id': animal_id}

if use_date_filter:
    restr = restr & f'session_tmst >= "{from_date}"'
    restr = restr & f'session_tmst <= "{to_date}"'
if use_session_filter:
    restr = restr & f'session >= {from_session}'
    restr = restr & f'session <= {to_session}'

sessions = (restr - exp.Session.Excluded).fetch('session', 'session_tmst')

In [ ]:
# ================================== Session summary across all sessions ==================================
# List to collect session info
session_summary = []
session_excluded = []

for session, session_tmst in zip(*sessions):
    
    # Key for this session
    key_session = {'animal_id': animal_id, 'session': session}

    task_rel = exp.Session.Task & key_session
    
    if len(task_rel):
        task_name_full = task_rel.fetch1('task_name')   # full path
    
        # -------- Detect neuropixels protocols --------
        neuropixels_paths = [
            r'Z:\scripts\pyconf\neuropixels',
            'mnt/lab/labstuff/scripts/pyconf/neuropixels'
        ]
    
        is_neuropixels = any(p in task_name_full for p in neuropixels_paths)
    
        task_full = task_name_full.split('/')[-1]
        protocol = os.path.splitext(task_full)[0]

    else:
        protocol = 'unknown'
        is_neuropixels = False

    
    
    # protocol = os.path.splitext(task_full)[0]  # removes ".py"  

    # find the setup
    setup = (exp.Session() & key_session).fetch('setup')


    # ------------------------- Neuropixels sessions -------------------------
    if is_neuropixels:
        session_summary.append({
            'animal_id': animal_id,
            'session_tmst': session_tmst,
            # 'session_tmst': pd.to_datetime(session_tmst).date(), # remove the time
            'session': session,
            'protocol': protocol,
            'rewarded_trials': np.nan,
            'punished_trials': np.nan,
            'valid_trials': np.nan,
            'aborted_trials': np.nan,
            'total_trials': np.nan,
            'session_perf': np.nan,
            'setup': setup,
            'is_neuropixels': True
        })
        continue

    # ------------------------- Trial counts -------------------------
    rewards = len(exp.Trial.StateOnset & key_session & 'state="Reward"')
    punishs = len(exp.Trial.StateOnset & key_session & 'state="Punish"')
    valid_trials = rewards + punishs
    session_perf = round((rewards / valid_trials),2) if valid_trials > 0 else np.nan  # exclude aborts

    aborts = len(exp.Trial.Aborted & key_session)
    total_trials = valid_trials + aborts  # including aborts

    #  Exclude sessions with too few trials 
    if valid_trials == 0:
        session_excluded.append({'animal_id': animal_id, 'session': session,'session_time':session_tmst, 'protocol': protocol})
        continue  # skip storing this session

    # ------------------------- Store results -------------------------
    session_summary.append({
        'animal_id': animal_id,
        'session_tmst': session_tmst,
        # 'session_tmst': pd.to_datetime(session_tmst).date(), # remove the time/
        'session': session,
        'protocol': protocol,
        'rewarded_trials': rewards,
        'punished_trials': punishs,
        'valid_trials': valid_trials,
        'aborted_trials': aborts,
        'total_trials': total_trials,
        'session_perf': f"{session_perf:.2f}",
        'setup': setup,
        'is_neuropixels': False
    })

# ------------------------- Convert to DataFrame -------------------------
for entry in session_excluded:
    x = f"Session {entry['session']}: protocol - {entry['protocol']}, (session_tmst: {entry['session_time']})"
    print(x)

def highlight_setup(row): 
    # behavioral recordings
    if row['setup'] == 'ef-rp167' or row['setup'] == 'ef-rp5':
        return ['background-color: limegreen'] * len(row)
    # passive recordings
    elif row['setup'] == 'ef-rp20' or row['setup'] == 'ef-master01' :
        return ['background-color: lightyellow'] * len(row)
    else:
        return [''] * len(row) # without recording

df_session_summary = pd.DataFrame(session_summary)

# keep values integers
int_cols = [
    'rewarded_trials',
    'punished_trials',
    'valid_trials',
    'aborted_trials',
    'total_trials',
]

df_session_summary[int_cols] = df_session_summary[int_cols].astype('Int64')


pd.set_option('display.max_rows', None)     # show all rows
pd.set_option('display.max_columns', None)  # optional: show all columns

# remove column "is_neuropixel"
df_session_summary = df_session_summary.drop(columns=['is_neuropixels'])


# Display the summary
display(df_session_summary.style.apply(highlight_setup, axis=1))
    

## Exclude sessions

In [ ]:
animal = animal_id
sessions_to_exclude = [582]  # list of sessions you want to exclude

for session in sessions_to_exclude:
    exp.Session.Excluded.insert1({'animal_id': animal, 'session': session, 'reason': "abort"}, skip_duplicates=True)

In [ ]:
#verify that you exlude the session

pd.set_option('display.max_colwidth', None)

data = pd.DataFrame(exp.Session.Excluded() & {'animal_id': animal})
data